<a href="https://colab.research.google.com/github/mbahramii/conveyor-stone-detector/blob/main/Stone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import numpy as np
from pathlib import Path

INPUT_VIDEO  = "/content/input.mp4"
OUTPUT_VIDEO = "/content/output.mp4"

cap = cv2.VideoCapture(INPUT_VIDEO)
fps    = int(cap.get(cv2.CAP_PROP_FPS))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

print(f"✅ ویدیو: {width}x{height} | {fps}fps | {total} فریم")

frame_count = 0
while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    h, w = img.shape[:2]

    # zone
    zone = np.array([
        [int(w * 0.62), int(h * 0.64)],
        [int(w * 0.98), int(h * 0.64)],
        [int(w * 0.98), int(h * 0.98)],
        [int(w * 0.62), int(h * 0.98)],
    ], dtype=np.int32)

    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(roi_mask, [zone], 255)

    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    blurred  = cv2.GaussianBlur(enhanced, (5, 5), 0)

    binary = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 51, 10)
    binary = cv2.bitwise_and(binary, binary, mask=roi_mask)

    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close, iterations=3)
    binary  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_open,  iterations=2)

    binary_inv = cv2.bitwise_not(binary)
    binary_inv = cv2.bitwise_and(binary_inv, binary_inv, mask=roi_mask)

    cnts, _ = cv2.findContours(binary_inv, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)

    vis = img.copy()
    cv2.polylines(vis, [zone], True, (0, 255, 255), 2)

    stone_found = False
    if cnts:
        best = max(cnts, key=cv2.contourArea)
        if cv2.contourArea(best) > 1000:
            rect = cv2.minAreaRect(best)
            box  = np.intp(cv2.boxPoints(rect))
            cv2.drawContours(vis, [box], 0, (255, 0, 0), 2)
            cv2.putText(vis, "Stone", (box[0][0], box[0][1]-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            stone_found = True

    status = "STONE ✓" if stone_found else "NO STONE"
    color  = (0, 255, 0) if stone_found else (0, 0, 255)
    cv2.putText(vis, status, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

    out.write(vis)
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"  ⏳ {frame_count}/{total} فریم پردازش شد...")

cap.release()
out.release()
print(f"\n✅ ویدیو ذخیره شد: {OUTPUT_VIDEO}")

In [ ]:
import cv2
import numpy as np

INPUT_VIDEO  = "/content/input.mp4"
OUTPUT_VIDEO = "/content/output_area.mp4"

cap = cv2.VideoCapture(INPUT_VIDEO)
fps    = int(cap.get(cv2.CAP_PROP_FPS))
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

print(f"✅ ویدیو: {width}x{height} | {fps}fps | {total} فریم")

frame_count = 0
while cap.isOpened():
    ret, img = cap.read()
    if not ret:
        break

    h, w = img.shape[:2]

    zone = np.array([
        [int(w * 0.62), int(h * 0.64)],
        [int(w * 0.98), int(h * 0.64)],
        [int(w * 0.98), int(h * 0.98)],
        [int(w * 0.62), int(h * 0.98)],
    ], dtype=np.int32)

    roi_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(roi_mask, [zone], 255)

    gray     = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe    = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    blurred  = cv2.GaussianBlur(enhanced, (5, 5), 0)

    binary = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 51, 10)
    binary = cv2.bitwise_and(binary, binary, mask=roi_mask)

    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close, iterations=3)
    binary  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_open,  iterations=2)

    binary_inv = cv2.bitwise_not(binary)
    binary_inv = cv2.bitwise_and(binary_inv, binary_inv, mask=roi_mask)

    cnts, _ = cv2.findContours(binary_inv, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)

    vis = img.copy()
    cv2.polylines(vis, [zone], True, (0, 255, 255), 2)

    stone_found = False
    if cnts:
        best = max(cnts, key=cv2.contourArea)
        area_px = cv2.contourArea(best)
        if area_px > 1000:
            rect = cv2.minAreaRect(best)
            box  = np.intp(cv2.boxPoints(rect))
            rw, rh = int(rect[1][0]), int(rect[1][1])

            cv2.drawContours(vis, [box], 0, (255, 0, 0), 2)


            cv2.putText(vis, f"Area: {int(area_px)} px2",
                        (box[0][0], box[0][1] - 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            cv2.putText(vis, f"Size: {rw}x{rh} px",
                        (box[0][0], box[0][1] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            stone_found = True

    status = "STONE" if stone_found else "NO STONE"
    color  = (0, 255, 0) if stone_found else (0, 0, 255)
    cv2.putText(vis, status, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

    out.write(vis)
    frame_count += 1
    if frame_count % 30 == 0:
        print(f"  ⏳ {frame_count}/{total} فریم...")

cap.release()
out.release()
print(f"\n✅ ذخیره شد: {OUTPUT_VIDEO}")